# Persistence — Interview Notes

Persistence is the core feature of LangGraph using which the state of the entire workflow is stored at a specific location (memory, SQLite, Postgres, etc.).

Imp Pointer → It can store the final as well as intermediate state values with the help of checkpointers.

__What is Checkpointing?__

Checkpointing is the process of saving a snapshot of the graph's state at a point in time.

In simple words: every time something changes in the graph, LangGraph takes a "photo" of the current state and saves it. This saved photo is called a checkpoint.

A checkpoint stores:

The current state values (all the data at that point)

The next node(s) that should run

Metadata — like which step number this is, and what caused this update

How Checkpointing Works — Super-Steps

LangGraph doesn't save the state after every tiny change. It saves state after every __super-step__.

What is a super-step? 

A super-step = a collection of a single / multiple nodes that forms a part of graph.

If only one node runs at a time → 1 node = 1 super-step.

If multiple nodes run in parallel (like in a Parallelization workflow) → all of them together = 1 super-step.

Simple flow:

Super-step 1 runs → state updates → checkpoint saved

Super-step 2 runs → state updates → checkpoint saved

Super-step 3 runs → state updates → checkpoint saved
...

So basically: run → update state → save checkpoint → repeat, until the graph reaches END.

This gives us a full history of checkpoints — one for every super-step — not just the final result.

__Threads (Very Important Concept)__

A thread is like a unique ID for one specific run/conversation of the graph.

Every time you invoke a graph with a thread_id, all checkpoints from that run get saved under that thread_id.

Different thread_id = completely separate, independent state/history.

Same thread_id = LangGraph will load the last saved checkpoint for that thread and continue from there.

__Why threads matter:__

They let one application handle multiple users/conversations at the same time without mixing up their states.

Example: user A's chat and user B's chat can run on the same graph but stay completely isolated, because they use different thread_ids.

__python__

config = {"configurable": {"thread_id": "1"}}

graph.invoke({"messages": [...]}, config)

If you call invoke again with the same thread_id, it picks up right where it left off (because it loads the last checkpoint of that thread).

__How to View the History of Changes__

LangGraph gives you a method to see every checkpoint that was ever saved for a thread:

__python__

list(graph.get_state_history(config))

This returns a list of all checkpoints (from the very first super-step to the latest), each containing:

The state at that point

The next node to run

Metadata (step number, etc.)

__To get just the latest/current state (not the full history):__

python

graph.get_state(config)

Each checkpoint also has a __checkpoint_id__, which uniquely identifies that exact version. 

You can pass this checkpoint_id back into the config to jump to (replay from) that exact past checkpoint — this is what makes Time Travel possible.

4 Advantages of Persistence (using Checkpointers)

1. Short-Term Memory

Since every super-step's state is saved under a thread_id, the graph remembers everything that happened earlier in that same thread — like remembering earlier messages in a conversation.

2. Time Travel

Because every super-step is checkpointed (not just the final one), you can go back to any earlier checkpoint using its checkpoint_id and:

Replay from that point, or
Fork into a new/alternate path from that point

This is useful for debugging — you can see exactly what the state looked like at each step.

3. Human-in-the-Loop (HITL)

Since state is saved after every super-step, the graph can be paused at a specific node, wait for a human to review/approve/edit something, and then resume from that exact saved checkpoint — instead of losing everything or starting over.

4. Fault Tolerant (Most Important)

If a node fails midway, you don't lose the whole run — the graph can resume from the last successfully saved checkpoint, instead of starting from START again. Persistence is what makes this recovery possible.

Quick Interview Summary

"LangGraph saves the graph's state after every super-step using checkpointers — this is called checkpointing. These checkpoints are grouped under a thread_id, which is what allows multiple independent conversations to run on the same graph without mixing up. Because we keep a full history of checkpoints (viewable with get_state_history), and not just the final result, we get four benefits for free: short-term memory, time travel, human-in-the-loop, and fault tolerance."

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI()

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the party? Because it knew it would be a "crust" favorite!',
 'explanation': 'This joke plays on words by using a pun. The word "crust" can refer to the outer layer of a pizza or a party favorite. By saying the pizza went to the party because it knew it would be a "crust" favorite, the joke is making a play on words to suggest that the pizza knew it would be a popular and well-liked food choice at the party. It\'s a light-hearted and playful way to incorporate food and humor into a joke.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? Because it knew it would be a "crust" favorite!', 'explanation': 'This joke plays on words by using a pun. The word "crust" can refer to the outer layer of a pizza or a party favorite. By saying the pizza went to the party because it knew it would be a "crust" favorite, the joke is making a play on words to suggest that the pizza knew it would be a popular and well-liked food choice at the party. It\'s a light-hearted and playful way to incorporate food and humor into a joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-b712-6cef-8002-4cdddc5257a8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:50:01.518786+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-abb7-65be-8001-4573867f91dc'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? Because it knew it would be a "crust" favorite!', 'explanation': 'This joke plays on words by using a pun. The word "crust" can refer to the outer layer of a pizza or a party favorite. By saying the pizza went to the party because it knew it would be a "crust" favorite, the joke is making a play on words to suggest that the pizza knew it would be a popular and well-liked food choice at the party. It\'s a light-hearted and playful way to incorporate food and humor into a joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-b712-6cef-8002-4cdddc5257a8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:50:01.518786+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f198c95-abb7-65be-8001-4573867f91dc'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the tortellini? \n\nBecause it was too saucy for them!',
 'explanation': 'This joke plays on the idea of "saucy" being a term used to describe someone who is too forward or flirtatious. In this case, the pasta sauce is being personified as being too saucy for the tortellini, causing them to break up. It\'s a light-hearted play on words that combines the literal meaning of pasta sauce with the figurative meaning of being overly assertive.'}

In [14]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the tortellini? \n\nBecause it was too saucy for them!', 'explanation': 'This joke plays on the idea of "saucy" being a term used to describe someone who is too forward or flirtatious. In this case, the pasta sauce is being personified as being too saucy for the tortellini, causing them to break up. It\'s a light-hearted play on words that combines the literal meaning of pasta sauce with the figurative meaning of being overly assertive.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-a101-6221-8002-9a742d924572'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:51:46.578868+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-92cd-6869-8001-d225bfbf818d'}}, tasks=(), interrupts=())

In [15]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the tortellini? \n\nBecause it was too saucy for them!', 'explanation': 'This joke plays on the idea of "saucy" being a term used to describe someone who is too forward or flirtatious. In this case, the pasta sauce is being personified as being too saucy for the tortellini, causing them to break up. It\'s a light-hearted play on words that combines the literal meaning of pasta sauce with the figurative meaning of being overly assertive.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-a101-6221-8002-9a742d924572'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-15T16:51:46.578868+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f198c99-92cd-6869-8001-d225bfbf818d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up